## Monitor Lock-in, wavelength, piezo and temperature over time with live plot

Interaktive Steuerung während `lif_monitor()` läuft (Plot-Fenster muss aktiv/fokussiert sein):

- `1` / `2` &nbsp;→&nbsp; Steuerungsmodus wählen: `1` = Piezo, `2` = Temperatur (aktueller Modus wird oben rechts im Plot angezeigt)
- `UP` / `DOWN` &nbsp;→&nbsp; aktiven Wert (Piezo-Spannung bzw. Temperatur-Sollwert) erhöhen/verringern
- `LEFT` / `RIGHT` &nbsp;→&nbsp; Schrittweite des aktiven Modus wechseln (Piezo: `v_steps`, Temperatur: `t_steps`)
- `q` / `Escape` &nbsp;→&nbsp; Monitoring beenden

Bei einer Temperaturänderung wird kurz (~1.5 s) gewartet, danach läuft die Messung normal weiter; 
die gestrichelte Linie im Temperatur-Subplot zeigt den aktuellen Sollwert, 
die durchgezogene Linie die tatsächlich gemessene Temperatur.

In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from managers.lif_avp import LIFManager
# import utils.scan_utils as su 
import utils.file_utils as fu


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
lm = LIFManager(silent=True)

In [ ]:
lm.connect_all()
meas_time = 2.1

In [ ]:
lm.laser_on()

In [ ]:
lm.wlm.average_off()
meas_time = 0.33

In [ ]:
lm.wlm.average_on()
meas_time = 2.1

### Steuerung starten

Vor dem Start: Plot-Fenster nach dem Öffnen anklicken, damit es Tastatureingaben empfängt.
`1`=Piezo, `2`=Temperatur, `UP`/`DOWN`=Wert ändern, `LEFT`/`RIGHT`=Schrittweite, `q`=Stop.

In [ ]:
# %matplotlib qt
%matplotlib QtAgg
### lm.wlm.average_off() => meas_time=0.32 s
### lm.wlm.average_off() => meas_time=2.1 s
data = lm.lif_monitor(
                max_points=200, # 
                meas_time=meas_time,
                pause_time=0.01,
                spec_line="Ar I",
                silent=True,
                )

In [ ]:
data['scan_data'].info()

In [ ]:
data['scan_data'].tail()

In [ ]:
lm.laser_off()
lm.disconnect_all()

In [ ]:
print(data['laser_state'])
print(data['scan_data'].head())

In [ ]:
%matplotlib inline

## Save data

In [ ]:
#### Make file name #####
data_dir = fu.make_data_dir(base_name="lif_monitor")
file_path = fu.make_data_file_name(
    data_dir=data_dir,
    base_name="test_no_plasma",
    extension="csv",
)
print(file_path)
####################################################

In [ ]:
#### make meta data ##############################
meta_data = data["laser_state"]
comment = {
    "comment": "first test of the function",
}
meta_data.update(comment)
print(meta_data)
#################################################

#### save data ################################
print(type(data["scan_data"]))
df = data["scan_data"]
fu.save_dataframe(df=df,
                  file_path=file_path,
                  metadata=meta_data,
                  sep="\t",
                  index=True,
                  silent=False,
                  )
###############################################